# **安裝繁中、日、韓、英文字型包，以利matplotlib做圖呈現**

In [ ]:
!apt-get -y install fonts-noto-cjk

# **清除matplotlib的字型快取(cache) (否則它會一直抓不到新安裝的字型)**

In [ ]:
import matplotlib as mpl
!rm -rf ~/.cache/matplotlib

In [ ]:
from pathlib import Path # 用來為API數據爬取後做存檔
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive # 掛載專案雲端資料夾，該資料夾存放本專案的.ipynb、csv檔。
drive.mount('/content/drive')

# **字型設定**

In [ ]:
import matplotlib.font_manager as fm

# 確認字型路徑（通常安裝在此）
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
prop = fm.FontProperties(fname=font_path)

# 設定全域字型 (或者在繪圖時單獨指定)
plt.rcParams['font.family'] = prop.get_name()
# 解決負號顯示問題
plt.rcParams['axes.unicode_minus'] = False

# **為數據來源(csv)製作Path物件**

In [ ]:
chart_csv_dir = Path("<google_drive_path_to>/02_Data/01_Spotify_chart_raw")
API_csv_dir = Path("<google_drive_path_to>/02_Data/02_Spotify_API_raw")

In [ ]:
today = pd.Timestamp.now().strftime('%Y-%m-%d')

# **讀取2026年、三個地區的最新版本榜單**

In [ ]:
for file_path in chart_csv_dir.glob('*.csv'):
  if 'global-weekly-2026' in file_path.name:
    print(file_path)
    df_global = pd.read_csv(file_path, encoding='utf-8')
  elif 'kr-weekly-2026' in file_path.name:
    print(file_path)
    df_kr = pd.read_csv(file_path, encoding='utf-8')
  elif 'jp-weekly-2026' in file_path.name:
    print(file_path)
    df_jp = pd.read_csv(file_path, encoding='utf-8')

In [ ]:
df_global.head()

# **資料清理：將uri獨立出來並去掉前綴字，以uri_cleaned作為日後識別資料用**

In [ ]:
df_global["uri_cleaned"] = df_global['uri'].str.replace("spotify:track:", "").str.strip()
df_jp["uri_cleaned"] = df_jp['uri'].str.replace("spotify:track:", "").str.strip()
df_kr["uri_cleaned"] = df_kr['uri'].str.replace("spotify:track:", "").str.strip()
df_global["region"] = "Global"
df_jp["region"] = "Japan"
df_kr["region"] = "South Korea"

# **資料分析：分析日、韓、global的歌曲來源、點閱率、歌手佔比**

In [ ]:
def region_src_analysis(df: pd.DataFrame, region_name: str, save_file_path: str):
  """分析某地區的熱門歌曲之歌曲來源(唱片公司、獨立製作人)的佔比。
  觀察市占率，並存成圓餅圖。"""
  #創建畫布與繪圖區
  fig, axe = plt.subplots(2 , 1, figsize = (20, 15), dpi=100)
  # 在上圖，畫上top 11-20中source各自佔比。
  df_latest_top11to20_src = df.loc[10:19, 'source'].value_counts()
  pie_data3 = list(df_latest_top11to20_src.values)
  pie_labels3 = list(df_latest_top11to20_src.index)

  axe[0].pie(pie_data3,
            labels = pie_labels3,
            autopct='%1.1f%%',
            startangle=90)
  axe[0].set_title(f'Source Percentage in {region_name} Top11-20 track')


  # 在下圖，顯示Top10中source各自佔比。
  df_latest_top10_src = df.head(10)['source'].value_counts()
  pie_data4 = list(df_latest_top10_src.values)
  pie_labels4 = list(df_latest_top10_src.index)

  axe[1].pie(pie_data4,
            labels = pie_labels4,
            autopct='%1.1f%%',
            startangle=90)
  axe[1].set_title(f'Source Percentage in {region_name} Top10 track')
  plt.show()
  fig.savefig(save_file_path, dpi=100)

In [ ]:
def region_stream_analysis(df: pd.DataFrame, region_name: str, save_file_path: str):
  """分析某地區的熱門歌曲之歌曲點閱率的佔比。
  觀察聽眾喜好，並存成圓餅圖。"""

  #創建畫布與繪圖區
  fig, axe = plt.subplots(3 , 1, figsize = (20, 15))

  # 在上圖，畫上top 10 , top 11-20, top20-200的streams
  df_latest_top10_track = df.head(10)['track_name']
  df_latest_top10 = df.head(10)['streams']
  df_latest_top11to20 = df.loc[10:19,'streams']
  df_latest_top11to20_track = df.loc[10:19,'track_name']
  df_latest_others = df.loc[20:, 'streams']

  pie_data = [df_latest_top10.sum(0), df_latest_top11to20.sum(0), df_latest_others.sum(0)]
  pie_labels = [f'Top10 in {region_name}', f'Top11-20 in {region_name}' , f'Top20-200 in {region_name}']
  axe[0].pie(pie_data,
            labels=pie_labels,
            autopct='%1.1f%%',
            startangle=90)
  axe[0].set_title(f'Streams Percentage btw {region_name} Top 10, 11-20 and 20-200')

  # 在中圖，顯示Top11-20中各自streams佔比。
  pie_data2 = df_latest_top11to20
  pie_labels2 = df_latest_top11to20_track
  explodes = [0.2] * 1 + [0.1] * 4 + [0] * 5
  axe[1].pie(pie_data2,
            labels = pie_labels2,
            autopct='%1.1f%%',
            startangle=90,
            explode = explodes)
  axe[1].set_title(f'Streams Percentage in {region_name} Top11-20 track')

  # 在下圖，顯示Top10中各自streams佔比。
  pie_data3 = df_latest_top10
  pie_labels3 = df_latest_top10_track
  explodes = [0.2] * 1 + [0.1] * 4 + [0] * 5
  axe[2].pie(pie_data3,
            labels = pie_labels3,
            autopct='%1.1f%%',
            startangle=90,
            explode = explodes)
  axe[2].set_title(f'Streams Percentage in {region_name} Top10 track')
  plt.show()
  fig.savefig(save_file_path, dpi=100)


In [ ]:
def region_artist_analysis(df: pd.DataFrame, region_name: str, save_file_path: str):
  """分析某地區的熱門歌曲之歌手、表演團體的佔比。
  觀察歌手的多產與叫座程度，並存成圓餅圖。"""

  #創建畫布與繪圖區
  fig, axe = plt.subplots(2 , 1, figsize = (10, 10), dpi=100)
  # 在上圖，畫上top 11-20中source各自佔比。
  df_latest_top11to20_src = df.loc[10:19, 'artist_names'].value_counts()
  pie_data3 = list(df_latest_top11to20_src.values)
  pie_labels3 = list(df_latest_top11to20_src.index)

  axe[0].pie(pie_data3,
            labels = pie_labels3,
            autopct='%1.1f%%',
            startangle=90)
  axe[0].set_title(f'Artists Percentage in {region_name} Top11-20 track')

  # 在下圖，顯示Top10中source各自佔比。
  df_latest_top10_src = df.head(10)['artist_names'].value_counts()
  pie_data4 = list(df_latest_top10_src.values)
  pie_labels4 = list(df_latest_top10_src.index)

  axe[1].pie(pie_data4,
            labels = pie_labels4,
            autopct='%1.1f%%',
            startangle=90)
  axe[1].set_title(f'Artists Percentage in {region_name} Top10 track')

  # 存檔
  plt.show()
  print()
  fig.savefig(save_file_path, dpi=100)

1. 分析global榜單

In [ ]:
save_to_1 = f"<google_drive_path_to>/03_Output/01_Figures/Global_source_analysis_{today}.png"
save_to_2 = f"<google_drive_path_to>/03_Output/01_Figures/Global_streams_analysis_{today}.png"
save_to_3 = f"<google_drive_path_to>/03_Output/01_Figures/Global_artist_analysis_{today}.png"
region_src_analysis(df_global, "global", save_to_1)
region_stream_analysis(df_global, 'global', save_to_2)
region_artist_analysis(df_global, 'global', save_to_3)

2. 分析japan榜單

In [ ]:
save_to_1 = f"<google_drive_path_to>/03_Output/01_Figures/Japan_source_analysis_{today}.png"
save_to_2 = f"<google_drive_path_to>/03_Output/01_Figures/Japan_streams_analysis_{today}.png"
save_to_3 = f"<google_drive_path_to>/03_Output/01_Figures/Japan_artist_analysis_{today}.png"
region_src_analysis(df_jp, "Japan", save_to_1)
region_stream_analysis(df_jp, 'Japan', save_to_2)
region_artist_analysis(df_jp, 'Japan', save_to_3)

3. 分析South Korea榜單

In [ ]:
save_to_1 = f"<google_drive_path_to>/03_Output/01_Figures/South_Korea_source_analysis_{today}.png"
save_to_2 = f"<google_drive_path_to>/03_Output/01_Figures/South_Korea_streams_analysis_{today}.png"
save_to_3 = f"<google_drive_path_to>/03_Output/01_Figures/South_Korea_artists_analysis_{today}.png"
region_src_analysis(df_kr, "South_Korea", save_to_1)
# region_stream_analysis(df_kr, 'South_Korea', save_to_2)
# region_artist_analysis(df_kr, 'South_Korea', save_to_3)

3. 分析South Korea榜單

In [ ]:
df_merged = pd.concat([df_global, df_jp, df_kr], axis=0)
df_merged["uri_cleaned"] = df_merged['uri'].str.replace("spotify:track:", "")
df_merged["uri_cleaned"] = df_merged["uri_cleaned"].str.strip()
df_merged

# **資料分析：分析歌曲是否能跨地區進榜，即，各地區的熱門歌曲之重疊度。**

In [ ]:
from matplotlib_venn import venn3 # import 文氏圖模組與venn3函式

In [ ]:
# 準備各個地區的歌曲集合
global_set = set(df_global["uri_cleaned"])
japan_set = set(df_jp["uri_cleaned"])
korea_set = set(df_kr["uri_cleaned"])

# 設定畫布與坐標軸系統
fig, axe = plt.subplots(1 , 1, figsize = (10, 8), dpi=80)
# plt.figure(figsize=(10, 8))

# 繪製文氏圖
v1 = venn3(subsets=[global_set, japan_set, korea_set],
          set_labels =('Global', 'Japan', 'South Korea'),
          set_colors=('blue','lightgreen','red'),
          ax=axe)
plt.title("Tracks Overlap in Global/Japan/South Korea (unit: tracks)", fontsize=16)
plt.legend(('in global','in jp','in jp and global','in kr','in kr and global','in jp and kr', 'in three regions'))

plt.show()

# 存檔
save_file_path = f"<google_drive_path_to>/03_Output/01_Figures/tracks_venns_analysis_{today}.png"
fig.savefig(save_file_path, dpi=100)

# **挖出在最新一週榜單中，在至少2個地區上榜的歌曲是哪些，並且整理出點閱率以及該首歌曲在同時期、三個地區的名次各自為何，若只有入兩個地區的榜，則第三區名次標記為NaN。**

In [ ]:
# 找出重疊區的歌曲是哪些。
common_tracks_in_jp_and_kr = japan_set & korea_set #uri的set
common_tracks_in_kr_and_glb = korea_set & global_set
common_tracks_in_jp_and_glb = japan_set & global_set
common_tracks_three = global_set & japan_set & korea_set


# 合併three資料表
df_three = pd.concat([df_global, df_jp, df_kr], axis=0).sort_values(by = "uri_cleaned", axis = 0)
common_tracks_info_in_three = df_three[df_three["uri_cleaned"].isin(common_tracks_three)]
result = common_tracks_info_in_three.groupby(by=
                            ["uri_cleaned", "track_name", "artist_names"]).agg(
                            sum_of_streams = ("streams", "sum"),
                            max_streams = ("streams", "max"),
                            min_streams = ("streams", "min")).sort_values(
                            "sum_of_streams", ascending=False).reset_index()
result["rank_in_global"] = result.merge(df_global[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result["rank_in_jp"] = result.merge(df_jp[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result["rank_in_kr"] = result.merge(df_kr[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result = result.sort_values(by=[
                            "rank_in_global"],ascending=[True], na_position="last", ignore_index=True)

result.to_csv(f"<google_drive_path_to>/03_Output/02_Tables/In_latest_wks_top_tracks_in_three_regions_{today}.csv", mode="w")
# result

In [ ]:
# 合併jp&kr的資料表
df_jp_kr = pd.concat([df_jp, df_kr], axis=0)
common_tracks_info_in_jp_and_kr = df_jp_kr[df_jp_kr["uri_cleaned"].isin(common_tracks_in_jp_and_kr)]
result_jp_kr = common_tracks_info_in_jp_and_kr.groupby(by=
                            ["uri_cleaned", "track_name", "artist_names"]).agg(
                            sum_of_streams = ("streams", "sum"),
                            max_streams = ("streams", "max"),
                            min_streams = ("streams", "min")).reset_index()

result_jp_kr["rank_in_global"] = result_jp_kr.merge(df_global[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_jp_kr["rank_in_jp"] = result_jp_kr.merge(df_jp[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_jp_kr["rank_in_kr"] = result_jp_kr.merge(df_kr[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_jp_kr = result_jp_kr.sort_values(by=[
                            "rank_in_global"],ascending=[True], na_position="last", ignore_index=True)

# # 存檔且秀出
result_jp_kr.to_csv(f"<google_drive_path_to>/03_Output/02_Tables/In_latest_wks_top_tracks_in_JpKr_{today}.csv", mode="w")
result_jp_kr

In [ ]:
# 合併kr&glb的資料表
df_kr_glb = pd.concat([df_kr, df_global], axis=0)
common_tracks_info_in_kr_and_glb = df_kr_glb[df_kr_glb["uri_cleaned"].isin(common_tracks_in_kr_and_glb)]
result_kr_glb = common_tracks_info_in_kr_and_glb.groupby(by=
                            ["uri_cleaned", "track_name", "artist_names"]).agg(
                            sum_of_streams = ("streams", "sum"),
                            max_streams = ("streams", "max"),
                            min_streams = ("streams", "min")).reset_index()
result_kr_glb["rank_in_global"] = result_kr_glb.merge(df_global[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_kr_glb["rank_in_jp"] = result_kr_glb.merge(df_jp[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_kr_glb["rank_in_kr"] = result_kr_glb.merge(df_kr[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_kr_glb = result_kr_glb.sort_values(by=[
                            "rank_in_global"],ascending=[True], na_position="last", ignore_index=True)
# 存檔且秀出
result_kr_glb.to_csv(f"<google_drive_path_to>/03_Output/02_Tables/In_latest_wks_top_tracks_in_GlbKr_{today}.csv", mode="w")
result_kr_glb

In [ ]:
# 合併jp&glb的資料表
df_jp_glb = pd.concat([df_jp, df_global], axis=0).sort_values(by = "uri_cleaned", axis = 0)
common_tracks_info_in_jp_and_glb = df_jp_glb[df_jp_glb["uri_cleaned"].isin(common_tracks_in_jp_and_glb)]
result_jp_glb = common_tracks_info_in_jp_and_glb.groupby(by=
                            ["uri_cleaned", "track_name", "artist_names"]).agg(
                            sum_of_streams = ("streams", "sum"),
                            max_streams = ("streams", "max"),
                            min_streams = ("streams", "min")).reset_index()
result_jp_glb["rank_in_global"] = result_jp_glb.merge(df_global[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_jp_glb["rank_in_jp"] = result_jp_glb.merge(df_jp[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]
result_jp_glb["rank_in_kr"] = result_jp_glb.merge(df_kr[["uri_cleaned", "rank"]], how="left", left_on="uri_cleaned", right_on="uri_cleaned")["rank"]

result_jp_glb = result_jp_glb.sort_values(by=[
                            "rank_in_global"],ascending=[True], na_position="last", ignore_index=True)
# 存檔且秀出
result_jp_glb.to_csv(f"<google_drive_path_to>/03_Output/02_Tables/In_latest_wks_top_tracks_in_GlbJp_{today}.csv", mode="w")
result_jp_glb

# **資料分析：回溯三個地區三個月的榜單紀錄。**
# 為最新一週，三個地區都有上榜的歌曲，分析它們在過往三個月的名次變化。

In [ ]:
# 定義地區清單
import os
regions = ['global', 'kr', 'jp']
all_dfs = []

for file_path in chart_csv_dir.glob('*.csv'):
    df = pd.read_csv(file_path, encoding='utf-8')
    file_name = os.path.basename(file_path).replace('.csv', '')
    date_in_file_name = '-'.join(file_name.split('-')[-3:])
    df['chart_date'] = pd.to_datetime(date_in_file_name)

    df['uri_cleaned'] = df['uri'].str.split(':').apply(lambda x: x[-1].strip())

    for region in regions:
      if region in file_name:
        df['region'] = region
        break

    all_dfs.append(df) # 將處理好的df物件先放到列表暫存

# 合併所有資料，得到所有地區三個月資料整併
df_history = pd.concat(all_dfs, ignore_index=True)

In [ ]:
df_history[(df_history["uri_cleaned"] == '53iuhJlwXhSER5J2IYYv1W') & (df_history['region'] == 'jp')]

In [ ]:
# 篩選出12首目標歌曲的歷史軌跡
df_common_tracks_3m_history = df_history[df_history['uri_cleaned'].isin(common_tracks_three)]

# 計算每首歌在各區"出現過幾週"
stay_duration = df_common_tracks_3m_history.groupby(by =
 ['uri_cleaned', 'track_name', 'region']).size().reset_index(name='actual_weeks_on_chart')

stay_duration.to_csv(f"<google_drive_path_to>/03_Output/02_Tables/Wks_on_chart_for_top_tracks_in_three_regions_{today}.csv", mode="w")

In [ ]:
df_common_tracks_3m_history

In [ ]:
# 準備熱圖矩陣：先以global地區的排名變化來建
# 建立一個 Pivot Table：橫軸是日期，縱軸是歌名，數值是排名
heatmap_data = df_common_tracks_3m_history[df_common_tracks_3m_history["region"] == "global"].pivot(
    index="track_name", columns="chart_date", values="rank"
)
heatmap_data.columns = [d.strftime("%Y/%m/%d") for d in heatmap_data.columns]

In [ ]:
heatmap_data

In [ ]:
import seaborn as sns

fig, axe = plt.subplots(3, 1, figsize=(20, 15))

# 繪製global熱圖：排名越小(越靠近1)顏色越深，不在榜上則顯示為空白(NaN)
sns.heatmap(heatmap_data, ax = axe[0], annot=True, fmt=".0f",
            cmap="YlGnBu_r", cbar_kws={'label': 'Rank'},
            xticklabels= False)
axe[0].set_title("Top 12 Overlapping Tracks: Global Chart Presence (Last 3 Months)", fontsize=16)

# 繪製12支歌曲在jp地區的熱圖
heatmap_data2 = df_common_tracks_3m_history[df_common_tracks_3m_history["region"] == "jp"].pivot(
    index="track_name", columns="chart_date", values="rank")
heatmap_data2.columns = [d.strftime('%Y/%m/%d') for d in heatmap_data2.columns]

sns.heatmap(heatmap_data2, ax = axe[1], annot=True, fmt=".0f",
            cmap="YlGnBu_r", cbar_kws={'label': 'Rank'},
            xticklabels= False)
axe[1].set_title("Top 12 Overlapping Tracks: Japan Chart Presence (Last 3 Months)", fontsize=16)


# 繪製12支歌曲在kr地區的熱圖
heatmap_data3 = df_common_tracks_3m_history[df_common_tracks_3m_history["region"] == "kr"].pivot(
    index="track_name", columns="chart_date", values="rank"
)
heatmap_data3.columns = [d.strftime('%Y/%m/%d') for d in heatmap_data3.columns]

sns.heatmap(heatmap_data3, ax = axe[2], annot=True, fmt=".0f", cmap="YlGnBu_r", cbar_kws={'label': 'Rank'})
axe[2].set_title("Top 12 Overlapping Tracks: South Korea Chart Presence (Last 3 Months)", fontsize=16)
plt.xlabel("Chart Week")
plt.ylabel("Track Name")
plt.xticks(rotation=30)

plt.show()
save_file_path = f"<google_drive_path_to>/03_Output/01_Figures/rank_of_tracks_heatmap_{today}.png"
fig.savefig(save_file_path, dpi=200)